# Agentic AI — A Hands-On Tutorial

https://colab.research.google.com/drive/10MZbXaYmIITSYd9l8OEZxZBrFUlMU04n

## What actually makes AI "agentic"?

A plain LLM call is a single round trip: you send text in, you get text out.
It has no way to check its work, look anything up, or take more than one step.

An **agent** is an LLM wired into a loop where it can:

1. **Reason** about what to do next
2. **Act** by calling a tool (a function your code executes)
3. **Observe** the result of that action
4. **Repeat** — using that new information to decide the next step — until it's
   satisfied it has a final answer

```
                ┌─────────────────────────────────┐
                │                                  │
                ▼                                  │
         ┌─────────────┐   "I need X"      ┌───────────────┐
         │              │ ───────────────► │                │
         │    Claude    │    tool_use       │  Your Tools    │
         │   (reason)   │                   │  (act on the   │
         │              │ ◄─────────────── │   real world)  │
         └─────────────┘   tool_result      └───────────────┘
                │
                │  "I'm done, here's the answer"
                ▼
           Final response
```

Everything below is really just this loop, made gradually more capable:
more tools, memory across turns, an explicit plan, and eventually several
agents cooperating instead of just one.


## Setup

Install the SDK and set your API key. If `ANTHROPIC_API_KEY` isn't already in
your environment, you'll be prompted for it (it won't be displayed or saved
anywhere by this notebook).


In [ ]:
!pip install anthropic --quiet

In [ ]:
import os
import json
import getpass
from anthropic import Anthropic

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Enter your Anthropic API key: ")

client = Anthropic()

# claude-sonnet-5: a good balance of intelligence, speed, and cost for this tutorial.
# Swap in "claude-haiku-4-5-20251001" for a faster/cheaper model, or
# "claude-opus-4-8" for maximum capability on harder tasks.
MODEL = "claude-sonnet-5"


---
## Part 1 — A Plain LLM Call (No Agency Yet)

This is the baseline: one request, one response, no memory, no tools, no
ability to act on anything outside its own training. Let's ask it something
an actual customer would ask a support bot.


In [ ]:
response = client.messages.create(
    model=MODEL,
    max_tokens=300,
    messages=[{"role": "user", "content": "Hi, can you check the status of my order ORD-1042?"}],
)

print(response.content[0].text)


Claude will (correctly) tell you it has no way to check that — it has no
connection to any order system, so it can't know. That's not a model
limitation you can prompt your way around; it's the actual gap tools exist
to close. No matter how capable the underlying model is, it cannot act on
*your* business's data until you give it a way to.


---
## Part 2 — Giving Claude a Tool

A tool is just a JSON schema describing a function: its name, what it does
(the `description` — Claude relies on this to decide *when* to use it), and
its expected inputs (`input_schema`).

**Crucially, Claude never executes the tool itself.** It only ever asks for
one, by returning a `tool_use` content block. Running the function and
returning the result back to Claude is entirely your code's job.

We'll learn this mechanic with something deliberately simple — a calculator —
before wiring in real business data in Part 3.


In [ ]:
import ast
import operator

_OPS = {
    ast.Add: operator.add, ast.Sub: operator.sub,
    ast.Mult: operator.mul, ast.Div: operator.truediv,
    ast.Pow: operator.pow, ast.USub: operator.neg,
}

def safe_eval(expression):
    \"\"\"Safely evaluate a basic arithmetic expression (+ - * / ** and parens only).\"\"\"
    node = ast.parse(expression, mode="eval").body

    def _eval(n):
        if isinstance(n, ast.Constant):
            return n.value
        if isinstance(n, ast.BinOp):
            return _OPS[type(n.op)](_eval(n.left), _eval(n.right))
        if isinstance(n, ast.UnaryOp):
            return _OPS[type(n.op)](_eval(n.operand))
        raise ValueError(f"Unsupported expression: {expression}")

    return _eval(node)

calculator_tool = {
    "name": "calculator",
    "description": "Evaluate a basic arithmetic expression, e.g. '249.99 * 0.85'. Supports + - * / ** and parentheses.",
    "input_schema": {
        "type": "object",
        "properties": {
            "expression": {"type": "string", "description": "The math expression to evaluate"}
        },
        "required": ["expression"],
    },
}

print(safe_eval("249.99 * 0.85"))  # sanity check


In [ ]:
response = client.messages.create(
    model=MODEL,
    max_tokens=300,
    tools=[calculator_tool],
    messages=[{"role": "user", "content": "What's 249.99 minus a 15% cut of that?"}],
)

for block in response.content:
    print(block)

print("\nstop_reason:", response.stop_reason)


Notice `stop_reason` is `"tool_use"`, not `"end_turn"` — Claude has *paused*
mid-task, waiting for a tool result. It handed back a `tool_use` block with an
`id`, the tool `name`, and the `input` it wants to call it with. It has not
answered the question yet.

To finish the round trip, we execute the tool ourselves and send the result
back as a new message — a `tool_result` block, tagged with the same `id` so
Claude knows which call it answers.


In [ ]:
tool_use_block = next(b for b in response.content if b.type == "tool_use")
result = safe_eval(tool_use_block.input["expression"])
print("Tool computed:", result)

follow_up = client.messages.create(
    model=MODEL,
    max_tokens=300,
    tools=[calculator_tool],
    messages=[
        {"role": "user", "content": "What's 249.99 minus a 15% cut of that?"},
        {"role": "assistant", "content": response.content},
        {"role": "user", "content": [
            {"type": "tool_result", "tool_use_id": tool_use_block.id, "content": str(result)}
        ]},
    ],
)

print(follow_up.content[0].text)


That round trip — **assistant asks for a tool → we run it → we hand back the
result as a user turn** — is the fundamental unit of everything an agent does.
Right now we did it once, by hand. Real tickets need this to happen an
unknown number of times in a row, which is exactly what Part 3 automates —
and where we start wiring in real business data instead of bare arithmetic.


---
## Part 3 — The Agent Loop

Instead of hand-wiring one round trip, we wrap it in a loop: keep calling
Claude and executing whatever tools it asks for, feeding results back, until
it stops asking for tools (`stop_reason != "tool_use"`) — meaning it believes
it has a final answer.

This loop *is* the agent. Everything from here on just adds more capability
around this same shape. Let's also stop pretending — real support tickets
reference real orders, so we need a tool that can look one up. This is a
mock in-memory dict standing in for a real orders database or API; the
tool-calling mechanics are identical either way.


In [ ]:
ORDERS_DB = {
    "ORD-1042": {
        "customer": "Maria Chen", "item": "Wireless Noise-Cancelling Headphones",
        "price": 249.99, "status": "Delivered", "order_date": "2026-06-10", "delivered_date": "2026-06-14",
    },
    "ORD-2033": {
        "customer": "Jordan Alvarez", "item": "Standing Desk Converter",
        "price": 189.50, "status": "In Transit", "order_date": "2026-07-18", "delivered_date": None,
    },
    "ORD-3011": {
        "customer": "Priya Nair", "item": "Espresso Machine",
        "price": 429.00, "status": "Delivered", "order_date": "2026-05-02", "delivered_date": "2026-05-06",
    },
}

def order_lookup(order_id):
    order = ORDERS_DB.get(order_id)
    if not order:
        return f"No order found with ID {order_id}."
    return order

order_lookup_tool = {
    "name": "order_lookup",
    "description": "Look up an order by its ID to get the customer, item, price, status, and dates.",
    "input_schema": {
        "type": "object",
        "properties": {"order_id": {"type": "string", "description": "e.g. 'ORD-1042'"}},
        "required": ["order_id"],
    },
}


In [ ]:
def run_agent_loop(user_message, tools, tool_functions, system=None, max_iterations=8, verbose=True):
    """
    Run Claude in a loop, executing any tools it requests, until it produces
    a final text answer (or we hit max_iterations as a safety net).
    \"\"\"
    messages = [{"role": "user", "content": user_message}]

    for i in range(max_iterations):
        response = client.messages.create(
            model=MODEL,
            max_tokens=1024,
            system=system,
            tools=tools,
            messages=messages,
        )
        messages.append({"role": "assistant", "content": response.content})

        if response.stop_reason != "tool_use":
            return "".join(b.text for b in response.content if b.type == "text")

        tool_results = []
        for block in response.content:
            if block.type != "tool_use":
                continue
            if verbose:
                print(f"  [iteration {i+1}] calling {block.name}({block.input})")
            try:
                fn = tool_functions[block.name]
                output = fn(**block.input)
            except Exception as e:
                # Never let a broken tool crash the whole agent — hand the
                # error back to the model so it can decide what to do next.
                output = f"Error running {block.name}: {e}"
            tool_results.append({
                "type": "tool_result",
                "tool_use_id": block.id,
                "content": str(output),
            })
        messages.append({"role": "user", "content": tool_results})

    return "Stopped: max_iterations reached without a final answer."


In [ ]:
answer = run_agent_loop(
    user_message=(
        "I'd like to return order ORD-1042. It's an opened electronics item, so there's a "
        "15% restocking fee. How much would I get refunded?"
    ),
    tools=[order_lookup_tool, calculator_tool],
    tool_functions={
        "order_lookup": order_lookup,
        "calculator": lambda expression: safe_eval(expression),
    },
)
print("\nFinal answer:", answer)


Watch the printed iterations: Claude first looks up the order to find its
price, *then* uses that number in the calculator — two tool calls chained
together, each informed by the last, without us writing a single line of
task-specific logic. That adaptiveness is the whole point of the loop.


---
## Part 4 — Multiple Tools

Real agents usually have several tools available and need to pick the right
one — or combine several — for a given request. So far our refund math
assumed the return was automatically valid. In reality, returns are only
valid *within policy* — which means the agent needs to look up the actual
rule and reason about it, not just compute a number on request.


In [ ]:
POLICIES = {
    "return policy": (
        "Items may be returned within 30 days of the delivery date for a full refund, minus "
        "a 15% restocking fee for opened electronics. Items returned after 30 days are not "
        "eligible for a refund, but may qualify for store credit at our discretion."
    ),
    "shipping policy": (
        "Standard shipping takes 5-7 business days; expedited shipping takes 2-3 business days. "
        "Orders can only be redirected to a new address before their status becomes 'In Transit'."
    ),
    "warranty policy": (
        "All electronics come with a 1-year manufacturer's warranty covering defects, not "
        "accidental damage."
    ),
}

def policy_lookup(topic):
    topic = topic.lower()
    for key, value in POLICIES.items():
        if key in topic or topic in key:
            return value
    return "No matching policy found."

policy_lookup_tool = {
    "name": "policy_lookup",
    "description": "Look up our store's policy text on a topic: 'return policy', 'shipping policy', or 'warranty policy'.",
    "input_schema": {
        "type": "object",
        "properties": {"topic": {"type": "string"}},
        "required": ["topic"],
    },
}


In [ ]:
answer = run_agent_loop(
    user_message=(
        "Order ORD-3011 was delivered on 2026-05-06. Today is 2026-06-20, so it's been "
        "45 days. I want to return the espresso machine — am I within the return window, "
        "and if so, what's my refund after the restocking fee?"
    ),
    tools=[order_lookup_tool, policy_lookup_tool, calculator_tool],
    tool_functions={
        "order_lookup": order_lookup,
        "policy_lookup": policy_lookup,
        "calculator": lambda expression: safe_eval(expression),
    },
)
print("\nFinal answer:", answer)


This is a deliberately awkward case: 45 days is *past* the 30-day window, so
the correct answer is "not eligible for a refund" — not a calculator result.
A good agent needs to look up the rule, apply it correctly, and know *not* to
just crunch numbers on request. That's a meaningfully different (and more
realistic) skill than routing a query to the right tool — it's routing a
query to the right tool *and checking whether the rule even applies*.


---
## Part 5 — Memory

`run_agent_loop` currently starts a brand-new conversation every time you call
it — it has no memory of anything said before. Real support systems need two
different kinds of memory:

1. **Conversation memory** — remembering earlier turns in *this* ticket
2. **Persistent memory** — remembering facts across tickets, even in a
   completely separate conversation days later (what a CRM is *for*)

First, a small `Agent` class that keeps its message history between calls, so
it can hold a multi-turn conversation instead of starting fresh every time.


In [ ]:
class Agent:
    def __init__(self, tools, tool_functions, system=None, max_iterations=8):
        self.tools = tools
        self.tool_functions = tool_functions
        self.system = system
        self.max_iterations = max_iterations
        self.messages = []  # conversation memory lives here, across calls

    def ask(self, user_message, verbose=True):
        self.messages.append({"role": "user", "content": user_message})

        for i in range(self.max_iterations):
            response = client.messages.create(
                model=MODEL,
                max_tokens=1024,
                system=self.system,
                tools=self.tools,
                messages=self.messages,
            )
            self.messages.append({"role": "assistant", "content": response.content})

            if response.stop_reason != "tool_use":
                return "".join(b.text for b in response.content if b.type == "text")

            tool_results = []
            for block in response.content:
                if block.type != "tool_use":
                    continue
                if verbose:
                    print(f"  [iteration {i+1}] calling {block.name}({block.input})")
                try:
                    output = self.tool_functions[block.name](**block.input)
                except Exception as e:
                    output = f"Error running {block.name}: {e}"
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": str(output),
                })
            self.messages.append({"role": "user", "content": tool_results})

        return "Stopped: max_iterations reached without a final answer."


Now, persistent memory: a simple CRM-style note store the agent can write to
and read from using tools, just like any other tool. Unlike `self.messages`,
this survives even when we start a brand new `Agent` — a brand new ticket —
later.


In [ ]:
customer_notes = {}  # stand-in for a real CRM

def save_customer_note(customer, note):
    customer_notes.setdefault(customer, []).append(note)
    return f"Saved note for {customer}: {note}"

def get_customer_notes(customer):
    notes = customer_notes.get(customer, [])
    return notes if notes else "No notes on file for this customer."

save_note_tool = {
    "name": "save_customer_note",
    "description": "Save a note on a customer's account for future reference (status, preferences, prior issues).",
    "input_schema": {
        "type": "object",
        "properties": {
            "customer": {"type": "string"},
            "note": {"type": "string"},
        },
        "required": ["customer", "note"],
    },
}

get_notes_tool = {
    "name": "get_customer_notes",
    "description": "Retrieve any saved notes on file for a customer.",
    "input_schema": {
        "type": "object",
        "properties": {"customer": {"type": "string"}},
        "required": ["customer"],
    },
}


In [ ]:
memory_tools = [order_lookup_tool, policy_lookup_tool, calculator_tool, save_note_tool, get_notes_tool]
memory_tool_functions = {
    "order_lookup": order_lookup,
    "policy_lookup": policy_lookup,
    "calculator": lambda expression: safe_eval(expression),
    "save_customer_note": save_customer_note,
    "get_customer_notes": get_customer_notes,
}

# Ticket #1 -- a support rep flags this customer as a VIP. Brand-new conversation.
ticket_1 = Agent(memory_tools, memory_tool_functions, system="Be concise and professional.")
print(ticket_1.ask(
    "This is Jordan Alvarez, order ORD-2033. Please note on their account that they're a VIP member."
))


In [ ]:
# Ticket #2 -- days later, a totally different conversation, no shared message history.
# VIP members get their restocking fee waived -- that rule lives in the system prompt below.
ticket_2 = Agent(
    memory_tools,
    memory_tool_functions,
    system=(
        "Be concise and professional. Before quoting any refund, check the customer's notes -- "
        "VIP members have their restocking fee waived."
    ),
)
print(ticket_2.ask(
    "New ticket from Jordan Alvarez. They want to return their Standing Desk Converter "
    "(order ORD-2033) and want a refund quote, assuming it's within the return window."
))


`ticket_2` shares none of `ticket_1`'s conversation — it's a different `Agent`
instance entirely — but it still gets the refund right (fee waived), because
that fact lives in `customer_notes`, outside the message list. This is the
same pattern real long-term-memory systems use: conversation history for
context *within* a session, and an external store (a database, a CRM, a
vector store) for anything that needs to survive across sessions.


---
## Part 6 — Planning

So far Claude decides its next tool call one step at a time, with no visible
plan. That's fine for a single-issue ticket, but real tickets are often
messier — a customer lists two or three unrelated problems in one message.
It helps to make the agent lay out an explicit, trackable plan *before* it
starts acting, so nothing gets dropped.

We'll add a simple todo-list tool and instruct the agent, via its system
prompt, to plan before it acts.


In [ ]:
todo_list = []

def add_tasks(tasks):
    todo_list.extend({"task": t, "done": False} for t in tasks)
    return f"Added {len(tasks)} task(s). Current list: {todo_list}"

def complete_task(task):
    for item in todo_list:
        if item["task"] == task:
            item["done"] = True
            return f"Marked done: {task}"
    return f"Task not found: {task}"

def list_tasks():
    return todo_list

add_tasks_tool = {
    "name": "add_tasks",
    "description": "Add a list of subtasks to the plan. Call this once, at the start, to lay out your whole plan.",
    "input_schema": {
        "type": "object",
        "properties": {
            "tasks": {"type": "array", "items": {"type": "string"}}
        },
        "required": ["tasks"],
    },
}

complete_task_tool = {
    "name": "complete_task",
    "description": "Mark a subtask from the plan as complete.",
    "input_schema": {
        "type": "object",
        "properties": {"task": {"type": "string"}},
        "required": ["task"],
    },
}

list_tasks_tool = {
    "name": "list_tasks",
    "description": "See the current plan and which subtasks are done.",
    "input_schema": {"type": "object", "properties": {}},
}


In [ ]:
todo_list.clear()  # reset for a clean demo run

planning_system = \"\"\"You are a careful support agent. For any multi-part ticket:
1. First call add_tasks ONCE with every subtask you'll need to do.
2. Work through the subtasks using the appropriate tools.
3. Call complete_task after finishing each one.
4. Only give your final answer once every task is marked complete.
\"\"\"

planning_tools = [order_lookup_tool, policy_lookup_tool, calculator_tool, add_tasks_tool, complete_task_tool, list_tasks_tool]
planning_tool_functions = {
    "order_lookup": order_lookup,
    "policy_lookup": policy_lookup,
    "calculator": lambda expression: safe_eval(expression),
    "add_tasks": add_tasks,
    "complete_task": complete_task,
    "list_tasks": list_tasks,
}

answer = run_agent_loop(
    user_message=(
        "Hi, I've got three things: (1) order ORD-2033 hasn't arrived and I'm getting worried, "
        "(2) I want a refund quote for ORD-3011 -- it was delivered 45 days ago, so let me know "
        "if I even qualify, and (3) what's your warranty policy on electronics in general?"
    ),
    tools=planning_tools,
    tool_functions=planning_tool_functions,
    system=planning_system,
    max_iterations=10,
)
print("\nFinal answer:", answer)
print("\nFinal plan state:", todo_list)


Now the plan is externally visible and trackable at every step — not just
implicit in Claude's reasoning — so a three-issue ticket like this one is far
less likely to have one issue silently dropped. This is the same basic idea
production coding agents use to track long, multi-step tasks without losing
their place.


---
## Part 7 — Multi-Agent Orchestration (Planner + Workers)

A single agent juggling order tracking, refund policy, *and* general policy
questions tends to get unfocused as tickets grow — its system prompt has to
cover everything, and its tool list gets cluttered. The usual fix, and how a
real support org is organized in the first place: split responsibilities
across several agents.

- A **triage agent (orchestrator)** reads the ticket, breaks it into
  subtasks, and decides who should handle each one
- Several **specialist agents (workers)**, each narrowly scoped — its own
  system prompt, its own small set of tools, good at exactly one kind of job

The neat trick: delegation is just another tool call. The orchestrator gets
one tool, `delegate_to_worker`, and each time it uses it, we spin up an
entire `run_agent_loop` for that specialist, then hand the result back as the
`tool_result`. Agents calling agents, using the exact same loop we already
built.


In [ ]:
WORKERS = {
    "order_tracking_specialist": {
        "system": "You track order status and shipping. Use order_lookup for order details. Be concise.",
        "tools": [order_lookup_tool],
        "tool_functions": {"order_lookup": order_lookup},
    },
    "refunds_specialist": {
        "system": (
            "You handle refund and return-eligibility questions. Use order_lookup for order details, "
            "policy_lookup for the return policy, and calculator for any math. Always check eligibility "
            "against the policy before quoting a number. Be concise."
        ),
        "tools": [order_lookup_tool, policy_lookup_tool, calculator_tool],
        "tool_functions": {
            "order_lookup": order_lookup,
            "policy_lookup": policy_lookup,
            "calculator": lambda expression: safe_eval(expression),
        },
    },
    "policy_specialist": {
        "system": "You answer general policy questions (shipping, warranty, returns) using policy_lookup. Be concise.",
        "tools": [policy_lookup_tool],
        "tool_functions": {"policy_lookup": policy_lookup},
    },
}

def run_worker(worker_name, task):
    worker = WORKERS[worker_name]
    return run_agent_loop(
        user_message=task,
        tools=worker["tools"],
        tool_functions=worker["tool_functions"],
        system=worker["system"],
        verbose=False,
    )

def delegate_to_worker(worker_name, task):
    print(f"    -> triage delegating to [{worker_name}]: {task}")
    result = run_worker(worker_name, task)
    print(f"    <- [{worker_name}] returned: {result}\n")
    return result

delegate_tool = {
    "name": "delegate_to_worker",
    "description": (
        "Delegate a subtask to a specialist. Available specialists: "
        "'order_tracking_specialist' (shipping/order status), "
        "'refunds_specialist' (return eligibility and refund math), "
        "'policy_specialist' (general policy questions)."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "worker_name": {
                "type": "string",
                "enum": ["order_tracking_specialist", "refunds_specialist", "policy_specialist"],
            },
            "task": {"type": "string", "description": "A clear, self-contained subtask for that specialist"},
        },
        "required": ["worker_name", "task"],
    },
}


In [ ]:
triage_system = \"\"\"You are a support triage agent. Break the customer's ticket into subtasks and
delegate EVERY subtask to the single best-suited specialist using delegate_to_worker -- never look up
orders, policy, or do math yourself. Once you have all the results you need, combine them into one
clear, friendly final reply to the customer.\"\"\"

final_answer = run_agent_loop(
    user_message=(
        "Hi, my order ORD-2033 hasn't arrived yet and I'm getting worried -- can you check on it? "
        "Also, if I return my espresso machine (ORD-3011), which was delivered 45 days ago, do I "
        "still get a refund? And separately, what's your warranty policy on electronics?"
    ),
    tools=[delegate_tool],
    tool_functions={"delegate_to_worker": delegate_to_worker},
    system=triage_system,
    max_iterations=6,
)

print("=" * 60)
print("FINAL REPLY TO CUSTOMER:\n")
print(final_answer)


Notice the triage agent never touches `order_lookup` or `policy_lookup`
directly — it only ever calls `delegate_to_worker`, and each specialist runs
its *own* independent loop with its own tools and its own focused system
prompt. This is the same shape used in production multi-agent systems: a
thin coordination layer on top, and small, specialized agents doing the
actual work underneath — which is also, not coincidentally, how a real
support team is organized.

From here, this pattern scales further in fairly obvious directions:

- More specialists (a billing-disputes agent, an escalation agent for angry
  customers, a fraud-review agent)
- Specialists that themselves delegate to sub-specialists (nested
  orchestration)
- Running independent specialist calls concurrently instead of one at a time
